# Algoritmos de búsqueda con oráculo: Grover

Grover busca un elemento marcado entre `N` elementos sin ordenar, con una ventaja cuadrática sobre cualquier búsqueda clásica: donde clásicamente hacen falta del orden de `N` consultas al oráculo, Grover necesita del orden de `√N`. Este notebook construye el caso más pequeño con ventaja real: 2 qubits, 4 elementos, resuelto con una sola iteración.

## El oráculo de Grover

A diferencia del oráculo de Deutsch-Jozsa, que acumula `f(x)` en un qubit auxiliar, el oráculo de Grover marca el elemento buscado directamente con un cambio de fase, sin qubit auxiliar: `|x⟩ → -|x⟩` si `x` es el elemento marcado, `|x⟩ → |x⟩` en cualquier otro caso.

Para marcar `|11⟩` con 2 qubits, esa puerta es exactamente CZ: cambia la fase solo cuando los dos qubits valen 1.

## El difusor

Marcar el elemento no basta: su amplitud sigue siendo pequeña, indistinguible al medir. El difusor invierte todas las amplitudes respecto a su media, lo que aumenta la del elemento marcado y reduce las del resto. Se construye con H, X y otra vez la puerta que marca fase, aplicada esta vez sobre `|00⟩` en vez de sobre el elemento buscado.

In [ ]:
import polypus

qc = polypus.Circuit(2)
qc.h(0)
qc.h(1)  # superposicion uniforme sobre los 4 elementos

qc.cz(0, 1)  # oraculo: marca |11>

qc.h(0)  # difusor
qc.h(1)
qc.x(0)
qc.x(1)
qc.cz(0, 1)
qc.x(0)
qc.x(1)
qc.h(0)
qc.h(1)

qc.measure_all()

Con el patrón visto en `02b`:

In [ ]:
from qiskit import qasm2

qc_dibujo = qasm2.loads(
    qc.to_qasm2(), custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS
)
qc_dibujo.draw("mpl")

In [ ]:
result = polypus.run_quantum_circuit(qc, shots=1000, infrastructure="local")
print(result.counts[0])

El resultado es `'11'` el 100% de las veces: con una sola iteración de oráculo más difusor, y 4 elementos posibles, Grover encuentra el marcado con certeza. Con más elementos hacen falta más iteraciones, y deja de ser una certeza exacta para pasar a ser una probabilidad muy alta.

## Más allá de 2 qubits

El oráculo y el difusor de Grover para más de 2 qubits necesitan una puerta Z multi-controlada, que cambia la fase solo cuando todos los qubits de control valen 1 a la vez. Polypus no tiene ninguna puerta nativa de más de 2 qubits: `GateInstruction` solo define primitivas de 1 y 2, sin ninguna variante de control múltiple.

Hay dos formas de conseguir el mismo efecto. La primera es decomponer la puerta multi-controlada en las primitivas ya conocidas. Una Toffoli, control de 2 qubits sobre un tercero, se escribe con H, T, Tdg y CX:

In [ ]:
def ccx_decompuesta(qc, a, b, c):
    qc.h(c)
    qc.cx(b, c)
    qc.tdg(c)
    qc.cx(a, c)
    qc.t(c)
    qc.cx(b, c)
    qc.tdg(c)
    qc.cx(a, c)
    qc.t(b)
    qc.t(c)
    qc.cx(a, b)
    qc.h(c)
    qc.t(a)
    qc.tdg(b)
    qc.cx(a, b)

Se comprueba igual que cualquier otra puerta, contra las cuatro combinaciones de entrada de una Toffoli real:

In [ ]:
for a_val in (0, 1):
    for b_val in (0, 1):
        qc_prueba = polypus.Circuit(3)
        if a_val:
            qc_prueba.x(0)
        if b_val:
            qc_prueba.x(1)
        ccx_decompuesta(qc_prueba, 0, 1, 2)
        qc_prueba.measure_all()
        resultado = polypus.run_quantum_circuit(
            qc_prueba, shots=200, infrastructure="local"
        )
        print(f"a={a_val} b={b_val} ->", resultado.counts[0])

El qubit `c` solo se invierte cuando `a` y `b` valen 1 a la vez, exactamente el comportamiento de una Toffoli.

La segunda opción es pasar directamente por Qiskit, que sí tiene puertas multi-controladas, y dejar que Polypus ejecute ese circuito tal cual, como se vio en `02b`:

In [ ]:
from qiskit import QuantumCircuit

qc_qiskit = QuantumCircuit(3)
qc_qiskit.x(0)
qc_qiskit.x(1)
qc_qiskit.mcx([0, 1], 2)
qc_qiskit.measure_all()

resultado = polypus.run_quantum_circuit(qc_qiskit, shots=200, infrastructure="local")
print(resultado.counts[0])

El resultado es idéntico: `'111'`, con `a=1` y `b=1`.

## Resumen

Este notebook ha construido Grover para 2 qubits, encontrando el elemento marcado con una sola iteración y certeza total, y ha mostrado las dos formas de extender el oráculo y el difusor más allá de 2 qubits, dado que Polypus no tiene puertas multi-controladas nativas: decomponer a mano con H, T, Tdg y CX, o apoyarse en Qiskit.

La siguiente sección entrena circuitos parametrizados con datos, empezando por QAOA sobre el problema MaxCut.

## Siguiente paso

Continúa con: [`06a_entrenamiento_variacional.ipynb`](06a_entrenamiento_variacional.ipynb).